In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import os
from tensorflow import keras


class EyebrowDeepfakeDetector:
    def __init__(self, model_type='lstm', input_shape=(56,7), num_classes=1):
        self.model_type = model_type
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.model = self._build_model()
        self.scaler = StandardScaler()

    def _build_model(self):
        """Build and compile the model based on specified architecture type"""            
        if self.model_type == 'cnn':
            model = tf.keras.Sequential([
                tf.keras.layers.Input(shape=self.input_shape),
                tf.keras.layers.Conv1D(64, 3, activation='relu', padding='same'),
                tf.keras.layers.BatchNormalization(),
                tf.keras.layers.MaxPooling1D(2),
                tf.keras.layers.Conv1D(128, 3, activation='relu', padding='same'),
                tf.keras.layers.BatchNormalization(),
                tf.keras.layers.GlobalAveragePooling1D(),
                tf.keras.layers.Dense(64, activation='relu'),
                tf.keras.layers.Dropout(0.3),
                tf.keras.layers.Dense(1, activation='sigmoid')
            ])
        elif self.model_type=="lstm":
            model = tf.keras.Sequential([
                tf.keras.layers.Input(shape=self.input_shape),
                tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True)),
                tf.keras.layers.BatchNormalization(),
                tf.keras.layers.SpatialDropout1D(0.3),
                tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
                tf.keras.layers.BatchNormalization(),
                tf.keras.layers.Dense(16, activation='relu'),
                tf.keras.layers.Dropout(0.2),
                tf.keras.layers.Dense(1, activation='sigmoid')
            ])

        elif self.model_type=="model-3":
            from keras import layers,models,regularizers
            inputs = tf.keras.Input(shape=self.input_shape)

            x = layers.Conv1D(128, kernel_size=5, padding="same", activation="relu")(inputs)
            x = layers.BatchNormalization()(x)
            x = layers.MaxPooling1D(pool_size=2)(x)

            x = layers.Conv1D(128 * 2, kernel_size=3, padding="same", activation="relu")(x)
            x = layers.BatchNormalization()(x)
            x = layers.MaxPooling1D(pool_size=2)(x)

            # BiLSTM
            x = layers.Bidirectional(layers.LSTM(128, return_sequences=False))(x)
            x = layers.BatchNormalization()(x)

            # Dense layers
            x = layers.Dropout(0.5)(x)
            x = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
            x = layers.Dropout(0.5)(x)

            outputs = layers.Dense(1, activation="sigmoid")(x)

            model = models.Model(inputs, outputs)
        
        elif self.model_type=="model-4":
            from keras import layers,models,regularizers
            inputs = tf.keras.Input(shape=self.input_shape)

            # --- BiLSTM ---
            x = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(inputs)
            x = layers.BatchNormalization()(x)
            x = layers.Dropout(0.5)(x)

            x = layers.Bidirectional(layers.LSTM(128))(x)
            x = layers.BatchNormalization()(x)
            x = layers.Dropout(0.4)(x)

            # --- ANN Head ---
            for units in [128,64]:
                x = layers.Dense(units, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
                x = layers.BatchNormalization()(x)
                x = layers.Dropout(0.4)(x)

            outputs = layers.Dense(1, activation='sigmoid')(x)

            model = models.Model(inputs, outputs)

        elif self.model_type=="model-5":
            from keras import layers,models,regularizers
            inputs = tf.keras.Input(shape=self.input_shape)

            # --- ANN TimeDistributed over Timesteps ---
            x = layers.TimeDistributed(layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4)))(inputs)
            x = layers.TimeDistributed(layers.BatchNormalization())(x)
            x = layers.TimeDistributed(layers.Dropout(0.4))(x)

            if len([64,64]) > 1:
                x = layers.TimeDistributed(layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4)))(x)
                x = layers.TimeDistributed(layers.BatchNormalization())(x)
                x = layers.TimeDistributed(layers.Dropout(0.4))(x)

            # --- LSTM Layer ---
            x = layers.LSTM(128, return_sequences=False)(x)
            x = layers.BatchNormalization()(x)
            x = layers.Dropout(0.4)(x)

            # --- Output ---
            x = layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
            x = layers.Dropout(0.4)(x)
            outputs = layers.Dense(1, activation='sigmoid')(x)

            model = models.Model(inputs, outputs)
        else:
            raise ValueError(f"Unknown model type: {self.model_type}")
            
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3,weight_decay=1e-4),
            loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),
            metrics=[
                tf.keras.metrics.BinaryAccuracy(name='accuracy'),
                tf.keras.metrics.AUC(name='auc'),
                tf.keras.metrics.Precision(name='precision'),
                tf.keras.metrics.Recall(name='recall')
            ]
        )
        return model

    def _positional_encoding(self, max_length=56, d_model=7):
        """Generate positional encoding for transformer"""
        positions = np.arange(max_length)[:, np.newaxis]
        depths = np.arange(d_model)[np.newaxis, :]/d_model
        
        angle_rates = 1 / (10000**depths)
        angle_rads = positions * angle_rates
        
        pos_encoding = np.zeros(angle_rads.shape)
        pos_encoding[:, 0::2] = np.sin(angle_rads[:, 0::2])
        pos_encoding[:, 1::2] = np.cos(angle_rads[:, 1::2])
        
        return tf.cast(pos_encoding[np.newaxis, ...], dtype=tf.float32)

    def load_data(self, csv_path):
        """Load and reshape eyebrow movement data"""
        df = pd.read_csv(csv_path, header=None)
        X = df.iloc[:, :-1].values.astype(np.float32)
        y = df.iloc[:, -1].values.astype(np.int32)
        
        # Reshape to (samples, 75, 3)
        if X.shape[1] == 392:  # Verify input dimensions
            X = X.reshape(-1, 56, 7)
        return X, y

    def preprocess_data(self, X, y, test_size=0.2):
        """Standardize features and split data"""
        # Stratified train-test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, stratify=y, random_state=42)
        
        # Feature-wise standardization
        orig_shape = X_train.shape
        self.scaler.fit(X_train.reshape(-1, 7))
        X_train = self.scaler.transform(X_train.reshape(-1, 7)).reshape(orig_shape)
        X_test = self.scaler.transform(X_test.reshape(-1, 7)).reshape(X_test.shape)
        
        return X_train, X_test, y_train, y_test

    def train(self, X_train, y_train, X_val, y_val, epochs=100):
        """Train model with early stopping and checkpointing"""
        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=15, restore_best_weights=True),
            tf.keras.callbacks.ModelCheckpoint(
                f'best_{self.model_type}_model.keras', save_best_only=True)
        ]
        
        history = self.model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=64,
            callbacks=callbacks,
            verbose=2
        )

        # Print final training metrics
        print("\nFinal Training Metrics:")
        for metric in history.history:
            if not metric.startswith('val_'):
                print(f"Training {metric}: {history.history[metric][-1]:.4f}")
        
        # Print final validation metrics
        print("\nFinal Validation Metrics:")
        for metric in history.history:
            if metric.startswith('val_'):
                print(f"{metric}: {history.history[metric][-1]:.4f}")

                # Analyze stopping reason
        best_epoch = np.argmin(history.history['val_loss']) + 1
        print(f"Best model found at epoch {best_epoch}")

        # Plot validation metrics trajectory
        plt.figure(figsize=(10, 6))
        plt.plot(history.history['val_loss'], label='Validation Loss')
        plt.axvline(x=best_epoch-1, color='r', linestyle='--', 
                label=f'Best Epoch ({best_epoch})')
        plt.title('Validation Loss Trajectory')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend()
        plt.show()

        return history

    def evaluate(self, X_test, y_test):
        """Comprehensive model evaluation"""
        # Quantitative metrics
        results = self.model.evaluate(X_test, y_test, verbose=1)
        metrics = dict(zip(self.model.metrics_names, results))
        
        # Debug: Print the metrics dictionary
        print("Metrics Dictionary:", metrics)
        
        # Qualitative analysis
        y_pred_prob = self.model.predict(X_test,verbose=0)
        y_pred = (y_pred_prob > 0.5).astype(int).flatten()
        
        # Calculate and print accuracy directly
        from sklearn.metrics import accuracy_score
        calc_accuracy = accuracy_score(y_test, y_pred)
        print(f"Manually calculated accuracy: {calc_accuracy:.4f}")

        print("\nClassification Report:")
        print(classification_report(y_test, y_pred))
        
        # Confusion matrix
        cm = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
        plt.title('Confusion Matrix')
        plt.colorbar()
        plt.xticks([0, 1], ['Real', 'Fake'])
        plt.yticks([0, 1], ['Real', 'Fake'])
        plt.xlabel('Predicted')
        plt.ylabel('True')
        plt.tight_layout()
        plt.savefig(f'{self.model_type}_confusion_matrix.png')
        plt.close()
        
        # ROC curve
        fpr, tpr, _ = roc_curve(y_test, y_pred_prob)
        roc_auc = auc(fpr, tpr)
        plt.figure(figsize=(10, 8))
        plt.plot(fpr, tpr, color='darkorange', lw=2, 
                label=f'ROC curve (AUC = {roc_auc:.3f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('ROC Curve')
        plt.legend(loc="lower right")
        plt.savefig(f'{self.model_type}_roc_curve.png')
        plt.close()
        
        # Ensure accuracy is in the metrics dictionary
        if 'accuracy' not in metrics:
            metrics['accuracy'] = calc_accuracy
        
        return metrics


# Usage example
if __name__ == "__main__":
    # Initialize detector - choose model type
    detector = EyebrowDeepfakeDetector(model_type='model-5', input_shape=(56, 7))  #modify the input and model accordingly
    
    # Load and preprocess data
    X, y = detector.load_data(os.path.join("","comprehensive_features-DeepFakeDetection_new.csv"))
    X_train, X_test, y_train, y_test = detector.preprocess_data(X, y)
    
    # Split validation set
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)
    
    # Train model
    print("Model architecture:")
    detector.model.summary()
    history = detector.train(X_train, y_train, X_val, y_val)
    
        # Plot training history
    import matplotlib.pyplot as plt

    plt.figure(figsize=(12, 4))

    # Plot loss
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Model Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    # Plot accuracy
    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Training Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Model Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.tight_layout()
    plt.savefig(f'{detector.model_type}_training_history.png')
    plt.show()
    plt.close('all')

    # Evaluate
    print("\nFinal evaluation:")
    test_metrics = detector.evaluate(X_test, y_test)

        # Print all metrics in a formatted way
    print("\nTest Metrics Summary:")
    for metric_name, metric_value in test_metrics.items():
        print(f"Test {metric_name}: {metric_value:.4f}")

    # Save final model
    detector.model.save(f'final_{detector.model_type}_model.keras')
